### Imports and path setup

In [14]:
import sys
sys.path.insert(0, "/home/jovyan")
sys.path.insert(0, "/home/jovyan/src")

import mlflow
from llm_client import generate, is_configured
from grandma_personas import PERSONAS, render
from src import evaluate_personas as ep

print("Gemini key configured: ", is_configured())

Gemini key configured:  True


### Pointing at MLflow and test one persona

In [2]:
mlflow.set_tracking_uri("http://mlflow:5000")
mlflow.set_experiment("Avo-Fridge_Prototyping")

fridge = "ovos, babaclhau, batata, cebola, azeite"

prompt = render("affectionate", fridge)
messages = [{"role": "user", "content": prompt}]

response, provider = generate(messages)
answer = response.choices[0].message.content

print(f"{provider}]\n{answer}")

gemini]
*Oh, meu querido!* Come here, let Avó see what we have! Let's see... ovos, ah, some lovely *bacalhau* that's already cooked, nice! And some good *batatas*, a little *cebola*, and of course, *azeite*! This is perfect for a little something warm and comforting.

You know, sometimes the best meals come from what's hiding in the fridge, *não é*? Don't you worry, we can make something delicious with these. *Ai que bom!*

Let's make a simple *bacalhau* and potato dish. It’s so easy, even you can do it, *filho*.

First, take those *batatas*. Peel them and cut them into nice, bite-sized pieces, like little cubes. Then, put them in a pot with some water and a pinch of salt, and let them cook until they are tender. Not too mushy, you understand? Just soft enough to be poked with a fork.

While the *batatas* are cooking, let's get the *bacalhau* ready. Flake it into smaller pieces. You can get rid of any bones, *coitadinho*, we don't want those.

Now, take that *cebola* and chop it up fin

### Log previous run to MLflow

In [4]:
with mlflow.start_run(run_name="affectionate-manual-test"):
    mlflow.log_param("persona", "affectionate")
    mlflow.log_param("provider", provider)
    mlflow.log_param("fridge_items", fridge)
    mlflow.log_text(answer, "response.txt")

2026/09/28 17:31:09 INFO mlflow.tracking._tracking_service.client: 🏃 View run affectionate-manual-test at: http://mlflow:5000/#/experiments/1/runs/df7c61144c1c4f3289188aed7edad528.
2026/09/28 17:31:09 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://mlflow:5000/#/experiments/1.


### Loop over the defined personas, to check the outputs

In [3]:
for name in PERSONAS:
    prompt = render(name, fridge)
    response, provider = generate([{"role": "user", "content": prompt}])
    answer = response.choices[0].message.content
    print(f"=== {name} [{provider}] ===\n{answer}\n")

=== affectionate [gemini] ===
*Ah, meu querido!* Come here, come here. Let's see what treasures we have hiding in this *frigorífico* today. *Nossa Senhora!* Look at all this! We have some lovely *ovos*, some beautiful *bacalhau* – oh, this will be so good! And *batatas*, *cebola*, and of course, our precious *azeite*.

Don't you worry about a thing, *filho*. We can make something *ai que bom!* with just these simple things. Let's make a lovely *Bacalhau com Batatas e Ovos*, just like my mother used to make. It’s so comforting, so full of flavor.

First, let's get those *batatas* ready. Peel them, *meu querido*, and cut them into nice, bite-sized pieces. Not too big, you want them to cook through nicely. Then, put them in a pan with some water and a little salt, and let them simmer until they are almost tender.

While they’re doing that, we’ll get the *bacalhau* sorted. If it’s salted *bacalhau*, you would have soaked it, of course. If it’s already ready to go, that’s even easier! Just 

### Evaluation

In [16]:
sys.argv = ["evaluate_personas", "--promote", "--rpm", "10"]
ep.main()

SystemExit: No GEMINI_API_KEY. Put it in docker/.env next to docker-compose.yml.
There is deliberately no offline mode: inventing scores would teach you to trust a number that measured nothing.

/opt/conda/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3707: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


### Check which off-topic case the strick persona failed to refuse.

- Loads OFF_TOPIC cases (non-food + prompt-injection attempts), the strict persona's raw template, looks_like_refusal (the same refusal-detector evaluate_personas.py uses for scoring), and generate (the Gemini/Ollama seam).  
- For each off-topic case, it renders the strict prompt with that case's query substituted in, calls the LLM directly, and checks whether the response reads as a refusal.  
- Prints OK or FAILED per case, and for any FAILED one, prints the first 200 characters of what it actually said — so you can see why it didn't refuse.

In [ ]:
from src.evaluation_set import OFF_TOPIC
from src.grandma_personas import render
from src.evaluate_personas import looks_like_refusal
from src.llm_client import generate

for case in OFF_TOPIC:
    prompt = render("strict", case["query"])
    response, provider = generate([{"role": "user", "content": prompt}])
    answer = response.choices[0].message.content
    refused = looks_like_refusal(answer)
    print(f"[{'OK' if refused else 'FAILED'}] {case['query'][:50]}")
    if not refused:
        print(f"   -> {answer[:200]}\n")